# Import Statements

In [43]:
import importlib
import session_organizer
from sentence_transformers import SentenceTransformer
import pandas as pd
importlib.reload(session_organizer)

<module 'session_organizer' from 'c:\\Users\\jdv223\\OneDrive - University of Kentucky\\Programming\\AI\\Session Creation Package\\session_organizer.py'>

## Step 1: Load and Examine Data

In [44]:
# First, examine the Excel file structure
file_path = "1.29.25 Abstracts.xlsx"
df_temp = pd.read_excel(file_path)

print("Available columns:")
for i, col in enumerate(df_temp.columns):
    print(f"{i}: {col}")

print(f"\nFile contains {len(df_temp)} rows and {len(df_temp.columns)} columns")
print("\nFirst few rows preview:")
print(df_temp.head())

Available columns:
0: Session
1: Submission Name
2: Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.
3: Submission ID - 7 digits
4: Technical Community
5: Profile: First Name
6: Profile: Last Name

File contains 1606 rows and 7 columns

First few rows preview:
                                             Session  \
0  Value-Added Chemicals Products and Materials t...   
1  Generative AI and Large Multimodal model for A...   
2  Generative AI and Large Multimodal model for A...   
3  Nutrient Removal, Recovery and Recycling: Manu...   
4    Erosion Control and Sediment Transport Research   

                                     Submission Name  \
0  Repurposing of low-value biomass into engineer...   
1  AI Tools and Text Embedding for Session Organi...   
2  Automatic ASABE AIM Session Creation using Mac...   
3  Soil and Wastewater Effluent Properties of Oil...   
4  MODELLING GULLIES 

In [45]:
# Define your column selections based on the output above
TITLE_COLUMN = 'Submission Name'  # Update based on your file
ABSTRACT_COLUMN = 'Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.'  # Update based on your file  
ID_COLUMN = 'Submission ID - 7 digits'  # Update based on your file

# Load the data using the session_organizer function
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations(
    file_path, 
    Title_name=TITLE_COLUMN,
    Abstract_name=ABSTRACT_COLUMN,
    Abstract_ID_name=ID_COLUMN
)

print(f"Loaded {len(df)} presentations successfully")
print(f"Title column: {title_column}")
print(f"Abstract column: {abstract_column}")
print(f"ID column: {abstract_id_column}")
print(f"Topic column: {topic_column}")

Loaded 1601 presentations successfully
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract


## Step 2: Load Embedding Model

In [46]:
# Available embedding models
EMBEDDING_MODELS = {
    "all-MiniLM-L6-v2": "sentence-transformers/all-MiniLM-L6-v2",
    "all-mpnet-base-v2": "sentence-transformers/all-mpnet-base-v2", 
    "paraphrase-MiniLM-L6-v2": "sentence-transformers/paraphrase-MiniLM-L6-v2",
    "cde-small-v1": "jxm/cde-small-v1"
}

# Select model (change as needed)
selected_model = "all-MiniLM-L6-v2"
model_name = EMBEDDING_MODELS[selected_model]

print(f"Loading embedding model: {model_name}")
embedding_model = SentenceTransformer(model_name, trust_remote_code=True)

if hasattr(embedding_model, 'model_card_data') and embedding_model.model_card_data:
    base_model = getattr(embedding_model.model_card_data, 'base_model', 'Unknown')
    print(f"Base model: {base_model}")
else:
    print(f"Model loaded: {model_name}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
Base model: sentence-transformers/all-MiniLM-L6-v2


# Process Steps

## Load Data

In [5]:
df, title_column, abstract_column, abstract_id_column, topic_column = session_organizer.load_presentations("1.29.25 Abstracts.xlsx", 
                                                                                         Title_name='Submission Name', 
                                                                                         Abstract_name='Abstract-Character max 4000-Abstracts will only be used to evaluate quality of talk and topic. They will not be published or able to be edited later.', 
                                                                                         Abstract_ID_name='Submission ID - 7 digits')

## Perform the Embedding

In [6]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', trust_remote_code=True)
print(f"Base model: {embedding_model.model_card_data.base_model}")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"Embeddings shape: {df_presentation_embeddings.shape}")
print(f"Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Base model: sentence-transformers/all-MiniLM-L6-v2


Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Embeddings shape: (1601, 385)
Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


In [7]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
df_presentation_similarities_old = session_organizer.calculate_similarity_matrix(embeddings_only, df, embedding_model)

In [5]:
# # Load the embeddings model
# embedding_model = SentenceTransformer('jxm/cde-small-v1', trust_remote_code=True)
# print(embedding_model.model_card_data.base_model)
# df_presentation_similarities, df_presentation_embeddings = embed_documents(df, topic_column, embedding_model)

## Remove Duplicates and Near-Duplicates

In [8]:
similarity_threshold = 0.99
# Remove near-duplicate presentations based on the similarity threshold
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
df, df_presentation_similarities_old, df_presentation_embeddings = session_organizer.remove_duplicates(df, df_presentation_similarities_old, df_presentation_embeddings, threshold=similarity_threshold)


Found 42 near-duplicate presentations to remove (keeping highest index).
Indices to remove: [41, 77, 78, 156, 221, 223, 269, 279, 325, 354, 410, 458, 476, 496, 540, 542, 560, 620, 712, 881, 904, 998, 1014, 1026, 1027, 1055, 1113, 1138, 1146, 1147, 1151, 1178, 1200, 1350, 1384, 1389, 1398, 1437, 1508, 1509, 1526, 1556]

Final number of oral presentations: 1559
Final shape of similarities matrix: (1559, 1559)
Final shape of embeddings matrix: (1559, 385)


## Create Sessions

In [10]:
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
df1, df_sessions1, labels1, metadata1 = session_organizer.create_sessions(df, df_presentation_similarities_old, embeddings_only, max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name="Session Code")
print(f"Created {metadata1['n_clusters']} sessions with a total of {metadata1['n_assigned_items']} presentations assigned.")
print(f"Unassigned presentations: {metadata1['n_unassigned_items']}")
print(metadata1)

Created 100 sessions with a total of 1559 presentations assigned.
Unassigned presentations: 0
{'n_clusters': 100, 'n_assigned_items': 1559, 'n_unassigned_items': 0, 'cluster_sizes': [19, 13, 15, 14, 18, 14, 11, 16, 15, 27, 11, 11, 14, 26, 13, 16, 13, 19, 16, 12, 15, 17, 22, 15, 20, 20, 17, 13, 16, 23, 14, 15, 19, 11, 13, 13, 16, 15, 17, 13, 11, 19, 10, 25, 15, 11, 17, 19, 38, 16, 25, 18, 16, 18, 14, 14, 12, 21, 15, 12, 15, 22, 15, 11, 16, 13, 16, 12, 19, 16, 15, 16, 16, 17, 12, 8, 10, 10, 24, 20, 14, 15, 12, 13, 8, 13, 17, 17, 11, 8, 9, 14, 24, 11, 16, 15, 17, 13, 13, 13], 'total_presentations': 1559, 'clustering_efficiency': 1.0}


In [29]:
df_presentations_updated, df_sessions_summary, labels_for_df_presentations, metadata2 = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings, max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name="Session Code")
# print(f"Created {metadata2['n_total_sessions']} sessions.")
# print(f"Hybrid sessions: {metadata2['n_hybrid_sessions_active']} with a total of {metadata2['n_general_presentations_assigned_to_generated']} general presentations assigned.")
print(metadata2)   

{'n_clusters': 100, 'n_assigned_items': 1559, 'n_unassigned_items': 0, 'cluster_sizes': [19, 13, 15, 14, 18, 14, 11, 16, 15, 27, 11, 11, 14, 26, 13, 16, 13, 19, 16, 12, 15, 17, 22, 15, 20, 20, 17, 13, 16, 23, 14, 15, 19, 11, 13, 13, 16, 15, 17, 13, 11, 19, 10, 25, 15, 11, 17, 19, 38, 16, 25, 18, 16, 18, 14, 14, 12, 21, 15, 12, 15, 22, 15, 11, 16, 13, 16, 12, 19, 16, 15, 16, 16, 17, 12, 8, 10, 10, 24, 20, 14, 15, 12, 13, 8, 13, 17, 17, 11, 8, 9, 14, 24, 11, 16, 15, 17, 13, 13, 13], 'total_presentations': 1559, 'clustering_efficiency': 1.0}


In [33]:
df_presentations3, df_sessions_summary3, labels_for_df_presentations3, metadata3 = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings,df_hybrid_presentations=df_hybrid_presentations,hybrid_session_column=session_col,df_hybrid_embeddings=df_hybrid_embeddings, max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name="Session Code")
# print(f"Created {metadata2['n_total_sessions']} sessions.")
# print(f"Hybrid sessions: {metadata2['n_hybrid_sessions_active']} with a total of {metadata2['n_general_presentations_assigned_to_generated']} general presentations assigned.")
print(metadata3)   

{'n_clusters': 100, 'n_assigned_items': 1559, 'n_unassigned_items': 0, 'cluster_sizes': [19, 13, 15, 14, 18, 14, 11, 16, 15, 27, 11, 11, 14, 26, 13, 16, 13, 19, 16, 12, 15, 17, 22, 15, 20, 20, 17, 13, 16, 23, 14, 15, 19, 11, 13, 13, 16, 15, 17, 13, 11, 19, 10, 25, 15, 11, 17, 19, 38, 16, 25, 18, 16, 18, 14, 14, 12, 21, 15, 12, 15, 22, 15, 11, 16, 13, 16, 12, 19, 16, 15, 16, 16, 17, 12, 8, 10, 10, 24, 20, 14, 15, 12, 13, 8, 13, 17, 17, 11, 8, 9, 14, 24, 11, 16, 15, 17, 13, 13, 13], 'total_presentations': 1559, 'clustering_efficiency': 1.0}


## Analyze Sessions

- session_coherence = "Are presentations within this session similar?" (internal session quality)
- session_distinctiveness = "Is this session's topic unique compared to others?" (relative session positioning)
- presentation_session_fit = "Does this presentation match the topic of others in the session?" (presentation fit)

Session Coherence measures cluster cohesion. It reflects how tighly grouped the topic of presentations within the session are.

Session Distinctiveness measures how unique each session's topic is. High values mean the session has a clear, focused theme that's different from other sessions. Low values suggest either the session mixes different topics or overlaps too much with other sessions.

Presentation-Session Fit is an individual presentations's average similarity to other presentation in its session. Generically, it can be referred to as "within_cluster_fit", "cluster_membership_strength", or "local_cohesion_score".

In [44]:
# Add to DataFrame
embeddings_only = df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
df_sessions['session_coherence'] = session_organizer.calculate_avg_similarity(df_sessions, df_presentation_similarities.values)
df_sessions['session_distinctiveness'] = session_organizer.calculate_silhouette_scores(df_sessions, embeddings_only, labels)
df['presentation_session_fit'] = session_organizer.calculate_document_similarities(df_presentation_similarities.values, labels)

## Create Session Titles & Keywords

In [9]:
# Test if Ollama is accessible
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"Ollama status: {response.status_code}")
    if response.status_code == 200:
        models = response.json()['models']
        print(f"Available models: {[m['name'] for m in models]}")
    else:
        print("Ollama server not responding correctly")
except Exception as e:
    print(f"Cannot connect to Ollama: {e}")
    print("Make sure to run 'ollama serve' first")

Ollama status: 200
Available models: ['llama3.2:latest']


In [10]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='ollama:llama3.2:latest')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Using model: llama3.2:latest
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 46.12 seconds
Average time per session: 15.37 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                     Ollama Title 1                                                             Ollama Title 2                                                                         Ollama Title 3                                                                                                                                         Ollama Keywords
          0 [133, 146, 210, 243, 439, 452, 596, 667, 778, 877, 941, 980, 1110, 1116, 1121, 1129, 1210, 1258, 1288]            19           0.516

In [11]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='llama-3.2-local')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

LLaMA model loaded successfully
Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 571.0409 seconds
Average time per session: 190.3470 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                                           Llama Title 1                                                            Llama Title 2                                                                                  Llama Title 3                                                                                                                                                                                                                                                                     

In [12]:
if 'model' in globals() or 'model' in locals():
    del model
    # Optionally, you can try to explicitly trigger garbage collection
    # import gc
    # gc.collect()
    print("LLaMA model has been flagged for unloading. Resources will be freed by the garbage collector.")
else:
    print("Model variable 'model' not found, or already unloaded.")

Model variable 'model' not found, or already unloaded.


In [13]:
# Test with first 3 sessions only
df_sessions_sample = df_sessions.head(3)
df_sessions_sample = session_organizer.generate_session_titles_and_keywords(df_sessions_sample, df, topic_column, model_name='gemini-2.0-flash')
# Generate titles and keywords for all sessions
# df_sessions = generate_session_titles_and_keywords(df_sessions, df, topic_column)

# Display sample results
print(df_sessions_sample.head().to_string(index=False))

Processing session 0 (1/3)...
  ✓ Generated titles for session 0
Processing session 1 (2/3)...
  ✓ Generated titles for session 1
Processing session 2 (3/3)...
  ✓ Generated titles for session 2

Total processing time: 8.5864 seconds
Average time per session: 2.8621 seconds
 cluster_id                                                                                   presentation_indices  cluster_size  session_coherence  session_distinctiveness                                                              Gemini Title 1                                                Gemini Title 2                                                                    Gemini Title 3                                                                            Gemini Keywords
          0 [133, 146, 210, 243, 439, 452, 596, 667, 778, 877, 941, 980, 1110, 1116, 1121, 1129, 1210, 1258, 1288]            19           0.516857                 0.031459                   AI-Powered Precision Systems for Crop and Weed Man

## Match Committees to Related Sessions

In [45]:
# Read the committee file from CSV/Excel with flexible column selection
committee_file_path = 'ASABE Committees.csv'  # Update this path as needed (can also use .xlsx)

# Load committees using the flexible function (similar to load_presentations)
df_committees, committee_name_column, description_column, combined_column = session_organizer.load_committees(
    committee_file_path,
    Committee_Name_column='Committee_Name',  # Actual column name in your file
    Description_column='Description',        # Actual column name in your file
    committee_name_column='Committee_Name',  # Desired output column name
    description_column='Description',        # Desired output column name
    combined_column='Name_Description'       # Combined column for embeddings
)

print(f"Loaded {len(df_committees)} committees")
print(f"Committee name column: {committee_name_column}")
print(f"Description column: {description_column}")
print(f"Combined column: {combined_column}")
print("\nFirst few committees:")
print(df_committees[[committee_name_column, description_column]].head())

# Generate embeddings for committees using the combined column
df_committee_embeddings = session_organizer.embed_documents(df_committees, combined_column, embedding_model)

Loaded 108 committees
Committee name column: Committee_Name
Description column: Description
Combined column: Name_Description

First few committees:
                                      Committee_Name  \
0  ASE-09 Environmental Quality Coordinating Comm...   
1                          ASE-12 Forest Engineering   
2  ASE-134 Fertilizers, Soil Conditioners & US TA...   
3              ASE-16 Engineering for Sustainability   
4  ASE-347 and US TAG TC 347 Data-driven agrifood...   

                                         Description  
0  Leads and coordinates the activities of ASABE ...  
1  Forested landscapes are essential for clean wa...  
2  US Technical Advisory Group for ISO TC 134. Le...  
3  ASE-16 leads and coordinates ASABE activities ...  
4  Standardization in the field of big-picture, d...  


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [46]:
df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']])
# Find the most similar committees for each session
session_committee_matches = session_organizer.find_most_similar_committees_by_presentations(
    df_sessions, 
    df_presentation_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']]), 
    df_committees, 
    df_committee_embeddings.drop(columns=[session_organizer.COLUMNS['EMBEDDING_MODEL']]), 
    top_n=3,
)

In [47]:
df_sessions = session_organizer.add_committee_matches_to_clusters(df_sessions, session_committee_matches)
# Display a sample of the results
print(f"\nSample of top committee matches:")
sample_columns = ['cluster_id', 'Top Committee Match', 'Top Committee Similarity', 
                  '2nd Committee Match', '2nd Committee Similarity', 
                  '3rd Committee Match', '3rd Committee Similarity']
print(df_sessions[sample_columns].head(10).to_string(index=False))


Sample of top committee matches:
 cluster_id                             Top Committee Match Top Committee Similarity                                    2nd Committee Match 2nd Committee Similarity                        3rd Committee Match 3rd Committee Similarity
          0               MS-45 Soil-Plant-Machine Dynamics                 0.430897                         NRES-244 Irrigation Management                 0.424146   PRS-702 Crop & Feed Processing & Storage                 0.422592
          1                  NRES-244 Irrigation Management                 0.484914                          MS-60 Unmanned Aerial Systems                  0.44994   PRS-702 Crop & Feed Processing & Storage                 0.437702
          2        PRS-702 Crop & Feed Processing & Storage                 0.307294     MS-23/19/3 Electronics for Identification (Animal)                 0.296466  MS-23/7 Harvest and US TAG ISO/TC 23/SC 7                 0.289872
          3 NRES-25 Streams, Reser

In [17]:
# # save the dataframes to pickle files
# df_sessions.to_pickle('df_sessions.pkl')
# df.to_pickle('df_presentations.pkl')

In [18]:
# df_sessions_sample.to_pickle('df_sessions_sample.pkl')

## Save Session Data for Later Use

After creating sessions, save the data so you can load it later for title generation without re-running the embedding process.

In [ ]:
# Save session data to pickle file for later use
import pickle

# Prepare session data dictionary
session_data = {
    'df': df,
    'df_sessions': df_sessions,
    'df_presentation_embeddings': df_presentation_embeddings,
    'df_presentation_similarities': df_presentation_similarities,
    'labels': labels,
    'metadata': metadata,
    'topic_column': topic_column,
    'original_filepath': file_path
}

# Save to pickle file
session_data_file = "session_data.pkl"
with open(session_data_file, 'wb') as f:
    pickle.dump(session_data, f)

print(f"✓ Session data saved to: {session_data_file}")
print(f"✓ Sessions: {len(df_sessions)}")
print(f"✓ Presentations: {len(df)}")

## Load Session Data (Alternative Start Point)

If you have saved session data, you can load it and continue from title generation.

In [ ]:
# Load previously saved session data
import pickle

session_data_file = "session_data.pkl"

try:
    with open(session_data_file, 'rb') as f:
        session_data = pickle.load(f)
    
    # Extract variables from loaded data
    df = session_data['df']
    df_sessions = session_data['df_sessions']
    df_presentation_embeddings = session_data['df_presentation_embeddings']
    df_presentation_similarities = session_data['df_presentation_similarities']
    labels = session_data['labels']
    metadata = session_data['metadata']
    topic_column = session_data['topic_column']
    
    print(f"✓ Session data loaded successfully!")
    print(f"✓ Sessions: {len(df_sessions)}")  
    print(f"✓ Presentations: {len(df)}")
    print(f"✓ Topic column: {topic_column}")
    print("\nYou can now proceed directly to title generation.")
    
except FileNotFoundError:
    print("✗ Session data file not found. Please run the full process first.")
except Exception as e:
    print(f"✗ Error loading session data: {e}")

## Complete Session Creation Workflow

This workflow is now separated into two main phases:
1. **Session Creation**: Embedding, clustering, and analysis (no LLM required)
2. **Title Generation**: Requires LLM configuration (Gemini API or Ollama)

In [ ]:
# Phase 1: Complete session creation workflow (no LLM needed)
# This includes all steps up to session analysis and saving intermediate results

print("=" * 50)
print("PHASE 1: SESSION CREATION")
print("=" * 50)

# Steps 1-6: Load data through session analysis
# (Use existing cells 5-18 for this phase)

# After completing session analysis, save intermediate results
print("\nPhase 1 complete - sessions created and analyzed")
print("Ready for Phase 2: Title Generation")

## Phase 2: Title Generation (Requires LLM Configuration)

This phase requires either:
- Gemini API key for online generation
- Ollama server running for local generation

In [ ]:
# Phase 2: Title generation - requires LLM setup
print("=" * 50)
print("PHASE 2: TITLE GENERATION")
print("=" * 50)

# Choose your LLM method
USE_GEMINI = True  # Set to False to use Ollama instead

if USE_GEMINI:
    # Requires GEMINI_API_KEY in .env file
    model_name = "gemini-2.0-flash"
    print("Using Gemini API for title generation...")
else:
    # Requires Ollama server running
    model_name = "ollama:llama3.2:latest"
    print("Using Ollama for title generation...")
    
    # Test Ollama connection
    import requests
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        if response.status_code == 200:
            print("✓ Ollama server is accessible")
        else:
            print("✗ Ollama server not responding properly")
    except Exception as e:
        print(f"✗ Cannot connect to Ollama: {e}")
        print("Please run 'ollama serve' first")

## Step 3: Create Embeddings (with Clean Progress Display)

In Jupyter notebooks, progress bars naturally update on the same line. The GUI application now mimics this behavior.

In [ ]:
print("Creating embeddings with live progress updates...")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"✓ Created embeddings with shape: {df_presentation_embeddings.shape}")
print(f"✓ Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

**Progress Display Behavior:**

- **Notebooks**: Progress bars update on the same line naturally
- **GUI Application**: Now mimics this behavior by updating the last line
- **Result**: Clean, real-time progress without log clutter

The progress will show as:
```
Batches:  47%|████████████████████                     | 24/51 [00:02<00:02, 10.33it/s]
```
And update in place until completion.

## Progress Display in GUI Application

The GUI application now uses a dual progress bar system for better user experience:

1. **Overall Progress Bar**: Shows progress through the main steps (1-6)
2. **Current Step Progress Bar**: Shows detailed progress within each step (especially useful for embedding operations)

This eliminates the need for text-based progress displays while providing clear visual feedback.

In [ ]:
# In notebooks, you still get the standard progress bars
print("Creating embeddings with standard notebook progress display...")
df_presentation_embeddings = session_organizer.embed_documents(df, topic_column, embedding_model)
print(f"✓ Created embeddings with shape: {df_presentation_embeddings.shape}")
print(f"✓ Embedding model used: {df_presentation_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

## GUI Application Layout

The Session Creation GUI has been redesigned with a logical two-column workflow:

**Left Column - Session Creation Workflow:**
1. File Selection
2. Column Selection  
3. Embedding Model
4. Session Creation (with Save button)

**Right Column - Session Naming Workflow:**
5. Load Existing Data (optional)
6. LLM Configuration
7. Generate Session Names

**Bottom Section (Full Width):**
- Dual Progress Bars (Overall + Current Step)
- Progress Log
- Exit Button

This layout provides a clear workflow: create sessions on the left, then name them on the right. The Load Data option at the top of the right column allows users to skip the session creation and jump directly to naming existing sessions.

## Load Hybrid Sessions (Alternative Data Source)

If you have existing hybrid sessions with pre-assigned presentations, you can load them directly instead of creating new sessions through clustering.

In [7]:
# Example: Load hybrid sessions from CSV/Excel file
hybrid_file_path = "Example Hybrid Session Invited Presentations.csv"  # Update this path as needed

# Load hybrid sessions using the flexible function
df_hybrid_presentations, df_hybrid_sessions, hybrid_session_col, title_col, abstract_col, abstract_id_col, topic_col = session_organizer.load_hybrid_sessions(
    hybrid_file_path,
    Session_column='Session',              # Actual column name in your file
    Title_column='Title',                  # Actual column name in your file  
    Abstract_column='Abstract',            # Actual column name in your file
    Abstract_ID_column='Submission ID - 7 digits',  # Actual column name in your file
    session_column='Session',              # Desired output column name
    title_column='Title',                  # Desired output column name
    abstract_column='Abstract',            # Desired output column name
    abstract_id_column='Abstract ID',      # Desired output column name
    topic_column='Title and Abstract'      # Combined column for embeddings
)

print(f"Loaded {len(df_hybrid_presentations)} hybrid presentations")
print(f"Session column: {hybrid_session_col}")
print(f"Title column: {title_col}")
print(f"Abstract column: {abstract_col}")
print(f"ID column: {abstract_id_col}")
print(f"Topic column: {topic_col}")

print("\nHybrid Sessions Summary:")
print(df_hybrid_sessions[[session_organizer.COLUMNS['CLUSTER_ID'], session_organizer.COLUMNS['SESSION_SIZE'], session_organizer.COLUMNS['HYBRID_SESSION_TITLE']]].to_string(index=False))

print("\nFirst few hybrid presentations with cluster mapping:")
print(df_hybrid_presentations[[hybrid_session_col, 'cluster_id', title_col]].head())

Loaded 6 hybrid presentations in 2 sessions
Session mapping: {'AI-Powered Remote Sensing for Crop and Soil Health Monitoring': 1, 'Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management': 2}
Loaded 6 hybrid presentations
Session column: Session
Title column: Title
Abstract column: Abstract
ID column: Abstract ID
Topic column: Title and Abstract

Hybrid Sessions Summary:
 cluster_id  session_size                                                                 hybrid_session_title
          1             3                        AI-Powered Remote Sensing for Crop and Soil Health Monitoring
          2             3 Open-Source “pyfao56” Evapotranspiration and Water Balance Tool for Water Management

First few hybrid presentations with cluster mapping:
                                             Session  cluster_id  \
0  AI-Powered Remote Sensing for Crop and Soil He...           1   
1  AI-Powered Remote Sensing for Crop and Soil He...           1   
2  AI-

In [8]:
# Generate embeddings for hybrid presentations (if needed for analysis)
df_hybrid_embeddings = session_organizer.embed_documents(df_hybrid_presentations, topic_col, embedding_model)
print(f"✓ Created embeddings for hybrid presentations with shape: {df_hybrid_embeddings.shape}")
print(f"✓ Embedding model used: {df_hybrid_embeddings[session_organizer.COLUMNS['EMBEDDING_MODEL']].iloc[0]}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Created embeddings for hybrid presentations with shape: (6, 385)
✓ Embedding model used: Unknown (sentence-transformers/all-MiniLM-L6-v2)


In [47]:
df_pres1, df_sessions1, labels1, metadata1 = session_organizer.create_sessions_w_hybrid(df, embedding_model.similarity, df_presentation_embeddings=df_presentation_embeddings,
                                                                                     df_hybrid_presentations=df_hybrid_presentations,
                                                                                     hybrid_session_column=hybrid_session_col, df_hybrid_embeddings=df_hybrid_embeddings,
                                                                                     max_sessions=100, min_session_size=8, tree_merge_stop=1, cluster_column_name="Session Code")
print(metadata1)
print(f"Created {metadata1['n_clusters']} sessions with {metadata1['n_assigned_items']} presentations.")
print(f"Unassigned Presentations: {metadata1['n_unassigned_items']}")
    

{'n_clusters': 100, 'n_assigned_items': 1601, 'n_unassigned_items': 0, 'n_total_items': 1601}
Created 100 sessions with 1601 presentations.
Unassigned Presentations: 0


In [27]:
from itertools import chain

all_indices = list(chain.from_iterable(df_sessions1['presentation_indices']))
max_index = max(all_indices)
min_index = min(all_indices)
unique_indices = set(all_indices)
expected_indices = set(range(min_index, max_index + 1))
missing_indices = sorted(expected_indices - unique_indices)

print(f"Max index: {max_index}")
print(f"Min index: {min_index}")
print(f"Number of unique indices: {len(unique_indices)}")
print(f"Missing indices: {missing_indices}")
print(f"Any indices skipped? {'Yes' if missing_indices else 'No'}")

Max index: 1605
Min index: 0
Number of unique indices: 1601
Missing indices: [441, 979, 1023, 1328, 1500]
Any indices skipped? Yes


## Working with Hybrid Sessions

Hybrid sessions loaded this way can be used for:
1. **Title generation** - Generate alternative titles using LLM models
2. **Committee matching** - Find relevant committees for each session
3. **Analysis** - Calculate session coherence and other metrics
4. **Comparison** - Compare with automatically created sessions

The `df_hybrid_sessions` DataFrame has the same structure as sessions created by `create_sessions()`, making them compatible with all downstream analysis functions.